Step 1: Start Spark

In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Initialize a Spark Session
spark = SparkSession.builder \
    .appName("Jupyter PySpark Setup") \
    .getOrCreate()

# Verify the session is working
print("Spark Version:", spark.version)


Spark Version: 4.1.2


Step 2: Load Data

In [5]:
#Load dataset
df=spark.read.csv("dataset.csv", header=True)

In [6]:
# it will show only first 20 rows of the dataset
df.show()

+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|   category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|    1001|       C001|        Alice| 25|Female|Electronics|  Headphones|       2| 1500| North|03-01-2025|           UPI|     4|
|    1002|       C002|          Bob| 32|  Male|   Clothing|     T-Shirt|       3|  700| South|03-01-2025|          Card|     5|
|    1003|       C003|      Charlie| 28|  Male|    Grocery|    Rice Bag|       1| 1200|  East|04-01-2025|           UPI|     4|
|    1004|       C004|        David| 45|  Male|Electronics|    Keyboard|       1| 1800|  West|05-01-2025|          Card|     5|
|    1005|       C005|          Eva| 22|Female|     Beauty|   Face Wash|       2|  350| North|05-01-2025

In [7]:
# WE can explicitly mention the number of rows we want to display
df.show(5)

+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|   category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|    1001|       C001|        Alice| 25|Female|Electronics|  Headphones|       2| 1500| North|03-01-2025|           UPI|     4|
|    1002|       C002|          Bob| 32|  Male|   Clothing|     T-Shirt|       3|  700| South|03-01-2025|          Card|     5|
|    1003|       C003|      Charlie| 28|  Male|    Grocery|    Rice Bag|       1| 1200|  East|04-01-2025|           UPI|     4|
|    1004|       C004|        David| 45|  Male|Electronics|    Keyboard|       1| 1800|  West|05-01-2025|          Card|     5|
|    1005|       C005|          Eva| 22|Female|     Beauty|   Face Wash|       2|  350| North|05-01-2025

In [8]:
# DIsplaying the name of columns
print(df.columns)

['order_id', 'customer_id', 'customer_name', 'age', 'gender', 'category', 'product_name', 'quantity', 'price', 'region', 'order_date', 'payment_method', 'rating']


In [9]:
# Displaying the data types of columns
# From here we can infer that, all the columns of the datset are of string type
# But columns like age, quantity, price, rating etc. should be in integer type.
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price: string (nullable = true)
 |-- region: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- rating: string (nullable = true)



In [ ]:
Step 3: Data Cleaning

In [14]:
# Deleting the duplicate rows
df=df.dropDuplicates()

In [15]:
from pyspark.sql.functions import col, sum

In [16]:
# Before dropping or filling the Null values, we will check how many null values are there in each column.
# If the missing values are less than 5% of the total data , then we prefer to use drop, as these do not much impact the data
# But if the missing values are more than this, we prefer to use fill.
# Therefore, checking how many null values are there in one column:

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+--------+-----------+-------------+---+------+--------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+--------+------------+--------+-----+------+----------+--------------+------+
|       0|          0|            0|  1|     0|       0|           0|       0|    1|     1|         0|             0|     0|
+--------+-----------+-------------+---+------+--------+------------+--------+-----+------+----------+--------------+------+



As missing data is less than 5% of total data , so we can use drop instead of fill

In [17]:
df=df.na.drop()

In [18]:
from pyspark.sql.functions import initcap

# Also, the column region contains inconsistent data i.e. east instead of East, west instead of West
#Therefore using initCap() function to capatalize first letter of the word in column region.
df = df.withColumn(
    "region",
    initcap(col("region"))
)

In [19]:
df.show()

+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|   category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|    1007|       C007|        Grace| 29|Female|    Grocery|        Milk|       5|   60|  East|06-01-2025|           UPI|     5|
|    1017|       C017|        Queen| 24|Female|     Beauty|     Shampoo|       2|  400| North|11-01-2025|          Cash|     3|
|    1038|       C038|         Luke| 41|  Male|   Clothing|     Sweater|       1| 1800| South|22-01-2025|          Card|     5|
|    1024|       C024|       Xavier| 40|  Male|Electronics|      Laptop|       1|55000|  West|15-01-2025|          Card|     5|
|    1005|       C005|          Eva| 22|Female|     Beauty|   Face Wash|       2|  350| North|05-01-2025

In [ ]:
Step 4: Filter Data

In [20]:
# Filtering based on Age
df.filter(col("age") > 30).show()

+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|   category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|    1038|       C038|         Luke| 41|  Male|   Clothing|     Sweater|       1| 1800| South|22-01-2025|          Card|     5|
|    1024|       C024|       Xavier| 40|  Male|Electronics|      Laptop|       1|55000|  West|15-01-2025|          Card|     5|
|    1006|       C006|        Frank| 34|  Male|   Clothing|       Jeans|       1| 1800| South|06-01-2025|          Cash|     4|
|    1031|       C031|        Emily| 31|Female|    Grocery|      Coffee|       1|  600|  East|18-01-2025|           UPI|     5|
|    1002|       C002|          Bob| 32|  Male|   Clothing|     T-Shirt|       3|  700| South|03-01-2025

In [21]:
# Filtering ased on category
df.filter(col("category") == "Electronics").show()

+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|   category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|    1024|       C024|       Xavier| 40|  Male|Electronics|      Laptop|       1|55000|  West|15-01-2025|          Card|     5|
|    1012|       C012|          Leo| 27|  Male|Electronics|     Speaker|       1| 3200|  West|09-01-2025|          Card|     4|
|    1008|       C008|        Harry| 41|  Male|Electronics|       Mouse|       2|  800|  West|07-01-2025|          Card|     4|
|    1036|       C036|        James| 38|  Male|Electronics| Smart Watch|       1|12000|  West|21-01-2025|          Card|     5|
|    1028|       C028|          Ben| 39|  Male|Electronics|      Tablet|       1|18000|  West|17-01-2025

In [22]:
# Filtering based on region
df.filter(col("region") == "North").show()

+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|   category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|    1017|       C017|        Queen| 24|Female|     Beauty|     Shampoo|       2|  400| North|11-01-2025|          Cash|     3|
|    1005|       C005|          Eva| 22|Female|     Beauty|   Face Wash|       2|  350| North|05-01-2025|           UPI|     3|
|    1037|       C037|        Kelly| 27|Female|     Beauty|  Face Cream|       1|  700| North|21-01-2025|           UPI|     4|
|    1029|       C029|        Clara| 24|Female|     Beauty| Nail Polish|       3|  200| North|17-01-2025|           UPI|     5|
|    1009|       C009|          Ivy| 26|Female|     Beauty|    Lipstick|       1|  500| North|07-01-2025

In [23]:
df.filter(col("region") == "South").show()

+--------+-----------+-------------+---+------+--------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+--------+------------+--------+-----+------+----------+--------------+------+
|    1038|       C038|         Luke| 41|  Male|Clothing|     Sweater|       1| 1800| South|22-01-2025|          Card|     5|
|    1006|       C006|        Frank| 34|  Male|Clothing|       Jeans|       1| 1800| South|06-01-2025|          Cash|     4|
|    1034|       C034|        Henry| 29|  Male|Clothing|      Hoodie|       1| 1600| South|20-01-2025|           UPI|     4|
|    1002|       C002|          Bob| 32|  Male|Clothing|     T-Shirt|       3|  700| South|03-01-2025|          Card|     5|
|    1030|       C030|       Daniel| 44|  Male|Clothing|    Trousers|       2| 1200| South|18-01-2025|          Card|     4|


In [ ]:
Step 5: Transform Data

In [24]:
# renaming column name from "customer_name" to "name"
df = df.withColumnRenamed(
    "customer_name",
    "name"
)

In [25]:
from pyspark.sql.types import IntegerType

# Casting the age and price column datatype from string to integer,
# so that we can perform aggregations on them.
df = df.withColumn(
    "age",
    col("age").cast(IntegerType())
)

df = df.withColumn(
    "price",
    col("price").cast(IntegerType())
)

In [26]:
df.show()

+--------+-----------+-------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|order_id|customer_id|   name|age|gender|   category|product_name|quantity|price|region|order_date|payment_method|rating|
+--------+-----------+-------+---+------+-----------+------------+--------+-----+------+----------+--------------+------+
|    1007|       C007|  Grace| 29|Female|    Grocery|        Milk|       5|   60|  East|06-01-2025|           UPI|     5|
|    1017|       C017|  Queen| 24|Female|     Beauty|     Shampoo|       2|  400| North|11-01-2025|          Cash|     3|
|    1038|       C038|   Luke| 41|  Male|   Clothing|     Sweater|       1| 1800| South|22-01-2025|          Card|     5|
|    1024|       C024| Xavier| 40|  Male|Electronics|      Laptop|       1|55000|  West|15-01-2025|          Card|     5|
|    1005|       C005|    Eva| 22|Female|     Beauty|   Face Wash|       2|  350| North|05-01-2025|           UPI|     3|
|    1006|       C006|  

In [ ]:
Step 6: Aggregation

In [27]:
# Printing total rows of dataFrame using .count() function
print("Total rows =", df.count())

Total rows = 37


In [28]:
from pyspark.sql.functions import avg
# Displaying the average price using avg() function
df.select(avg("price")).show()

+-----------------+
|       avg(price)|
+-----------------+
|4467.027027027027|
+-----------------+



In [29]:
from pyspark.sql.functions import min
# Calculating the min price from the dataFRame using min() function
df.select(min("price")).show()

+----------+
|min(price)|
+----------+
|        50|
+----------+



In [30]:
from pyspark.sql.functions import max
# Calculating the max price from the dataFRame using max() function
df.select(max("price")).show()

+----------+
|max(price)|
+----------+
|     55000|
+----------+



In [ ]:
Step 8: Group Data

In [31]:
# using groupby() to display the count of products in each category
df.groupBy(
    "category"
).count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    Grocery|   10|
|Electronics|   11|
|   Clothing|    9|
|     Beauty|    7|
+-----------+-----+



In [34]:
# Which region generates the highest revenue
from pyspark.sql.functions import sum

df.groupBy("region") \
  .agg(
      sum("price").alias("total_revenue")
  ) \
  .orderBy("total_revenue", ascending=False) \
  .show()

+------+-------------+
|region|total_revenue|
+------+-------------+
|  West|       143000|
| South|        12200|
| North|         5400|
|  East|         4680|
+------+-------------+



Region : West generates the highest revenue followed by South, north and East.

In [35]:
# Which Product category has the highest customers

from pyspark.sql.functions import count

df.groupBy("category") \
  .agg(
      count("*").alias("total_orders")
  ) \
  .orderBy("total_orders", ascending=False) \
  .show()

+-----------+------------+
|   category|total_orders|
+-----------+------------+
|Electronics|          11|
|    Grocery|          10|
|   Clothing|           9|
|     Beauty|           7|
+-----------+------------+



Electronics have the highest customer and is most poular category follwed by Grocery, Clothing, Beauty

In [36]:
# Average customer rating per category
from pyspark.sql.functions import avg

df.groupBy("category") \
  .agg(
      avg("rating").alias("avg_rating")
  ) \
  .orderBy("avg_rating", ascending=False) \
  .show()

+-----------+-----------------+
|   category|       avg_rating|
+-----------+-----------------+
|    Grocery|              4.7|
|Electronics|4.454545454545454|
|     Beauty|4.142857142857143|
|   Clothing|4.111111111111111|
+-----------+-----------------+



Grocery has the highest rating with 4.7

In [37]:
df.groupBy("category") \
  .agg(
      count("*").alias("orders"),
      sum("price").alias("revenue"),
      avg("rating").alias("rating")
  ) \
  .orderBy("revenue", ascending=False) \
  .show()

+-----------+------+-------+-----------------+
|   category|orders|revenue|           rating|
+-----------+------+-------+-----------------+
|Electronics|    11| 144500|4.454545454545454|
|   Clothing|     9|  12200|4.111111111111111|
|    Grocery|    10|   4680|              4.7|
|     Beauty|     7|   3900|4.142857142857143|
+-----------+------+-------+-----------------+



Last step:The above operations together form a pipeline.

In [44]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, initcap, count, sum, avg

# Create Spark Session
spark = SparkSession.builder \
    .appName("Retail Analytics Pipeline") \
    .getOrCreate()

# Load Data
df = spark.read.csv(
    "dataset.csv",
    header=True,
    inferSchema=True
)

# Data Cleaning
df = df.dropDuplicates()

df = df.fillna({
    "age": 0,
    "price": 0,
    "region": "Unknown"
})

# Transformation
df = df.withColumn("region", initcap(col("region"))) \
       .withColumnRenamed("customer_name", "name")

# Filtering
filtered_df = df.filter(
    (col("age") > 25) &
    (col("price") > 1000)
)

# Aggregation
result = filtered_df.groupBy("region").agg(
    count("*").alias("total_orders"),
    sum("price").alias("total_revenue"),
    avg("rating").alias("average_rating")
)

# Show results
result.show()

+------+------------+-------------+-----------------+
|region|total_orders|total_revenue|   average_rating|
+------+------------+-------------+-----------------+
| South|           6|        11100|4.166666666666667|
|  East|           2|         3000|              4.5|
|  West|           9|       142200|4.555555555555555|
+------+------------+-------------+-----------------+



In [46]:
# Convert Spark DataFrame to Pandas DataFrame
result_pd = result.toPandas()

In [48]:
# Save as CSV
result_pd.to_csv("results_new.csv", index=False)

print("results.csv created successfully!")

results.csv created successfully!


Built a Spark DataFrame pipeline to clean retail data,standardize regions, filter high-value transactions, and produce region-wise order count, revenue, and average rating metrics in results.csv